# 🛢️ YOLOv8n — Cylinder Detection with Ellipse Border

> **Lý do dùng YOLOv8n (detect) thay vì seg:**
> Dataset Roboflow `cylinder-eiuoi` chỉ có bounding box annotations (không có polygon mask).
> Thay vào đó, tại inference ta **fit ellipse** lên vùng phát hiện bằng GrabCut/contour —
> phù hợp hoàn toàn với hình dạng hình trụ (tiết diện tròn/elip).

**Pipeline:**
1. Cài packages
2. Mount Google Drive
3. Download dataset Roboflow → kiểm tra annotations
4. Xem mẫu ảnh
5. Train `yolov8n.pt`
6. Validate & biểu đồ
7. Inference — vẽ **ellipse border** (không bounding box)
8. Test ảnh upload
9. Test video upload
10. Export & download

---
⚡ **Runtime:** `Runtime > Change runtime type > T4 GPU`

## ⚙️ Cell 1 — Kiểm tra GPU & cài packages

In [ ]:
import subprocess, sys

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU detected:')
    # Show GPU name + memory only
    for line in result.stdout.split('\n'):
        if 'Tesla' in line or 'RTX' in line or 'GTX' in line or 'A100' in line or 'T4' in line or 'MiB' in line:
            print(' ', line)
else:
    print('⚠️  No GPU — training will be slow! Go to Runtime > Change runtime type > GPU')

print('\n📦 Installing dependencies ...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'ultralytics', 'roboflow', '--quiet', '--upgrade'],
    check=True
)
print('✅ Done!')

## 💾 Cell 2 — Mount Google Drive (tuỳ chọn)

In [ ]:
SAVE_TO_DRIVE    = True                             # ← False nếu không dùng Drive
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/CylinderDet'

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os; os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print(f'✅ Drive mounted. Output → {DRIVE_OUTPUT_DIR}')
else:
    DRIVE_OUTPUT_DIR = '/content'
    print('ℹ️  Không dùng Drive. Output lưu trong /content/')

## 📥 Cell 3 — Download dataset từ Roboflow & kiểm tra annotations

In [ ]:
from roboflow import Roboflow
import os, yaml

ROBOFLOW_API_KEY = 'HwZYq22onvjG9WU7kJSZ'
ROBOFLOW_WS      = 'arturo-perez-fzrw0'
ROBOFLOW_PROJECT = 'cylinder-eiuoi'
ROBOFLOW_VERSION = 1
DATASET_DIR      = '/content/cylinder_dataset'

print('🔽 Downloading dataset from Roboflow ...')
rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WS).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)

# Download format yolov8 (bounding box)
dataset = version.download('yolov8', location=DATASET_DIR, overwrite=True)

# Tìm data.yaml
DATA_YAML = None
for root, _, files in os.walk(DATASET_DIR):
    for f in files:
        if f.endswith('.yaml'):
            DATA_YAML = os.path.join(root, f)
            break
assert DATA_YAML, f'data.yaml not found under {DATASET_DIR}'

# Đọc metadata
with open(DATA_YAML) as f:
    meta = yaml.safe_load(f)
CLASS_NAMES = meta.get('names', [])
NUM_CLASSES = meta.get('nc', len(CLASS_NAMES))

print(f'\n✅ Dataset ready!')
print(f'   data.yaml  → {DATA_YAML}')
print(f'   Classes    : {NUM_CLASSES} → {CLASS_NAMES}')

# ── Kiểm tra loại annotations ────────────────────────────────────────────────
IMG_EXTS = {'.jpg','.jpeg','.png','.bmp','.webp'}
HAS_SEG  = False

for split in ['train', 'valid', 'test']:
    img_d = os.path.join(DATASET_DIR, split, 'images')
    lbl_d = os.path.join(DATASET_DIR, split, 'labels')
    if not os.path.exists(img_d):
        continue
    n_imgs = sum(1 for f in os.listdir(img_d) if os.path.splitext(f)[1].lower() in IMG_EXTS)
    print(f'   [{split:5s}] {n_imgs} images')

    # Kiểm tra xem label có polygon (seg) không
    if os.path.exists(lbl_d):
        for lf in os.listdir(lbl_d):
            lpath = os.path.join(lbl_d, lf)
            with open(lpath) as lfile:
                for line in lfile:
                    parts = line.strip().split()
                    if len(parts) > 5:   # >5 tokens → có polygon coordinates
                        HAS_SEG = True
                        break
            if HAS_SEG:
                break

print(f'\n🔎 Annotation type: {"SEGMENTATION (polygon)" if HAS_SEG else "DETECTION (bounding box only)"}')
if not HAS_SEG:
    print('   ℹ️  Dataset chỉ có bounding box.')
    print('   ✅ Sẽ train YOLOv8n (detect) + vẽ ELLIPSE BORDER tại inference.')
    MODEL_TYPE = 'detect'
else:
    print('   ✅ Dataset có polygon mask — có thể dùng YOLOv8n-seg.')
    MODEL_TYPE = 'segment'

BASE_MODEL = 'yolov8n-seg.pt' if MODEL_TYPE == 'segment' else 'yolov8n.pt'
print(f'   Base model : {BASE_MODEL}')

## 🖼️ Cell 4 — Xem mẫu ảnh dataset

In [ ]:
import os, random
import cv2, numpy as np
import matplotlib.pyplot as plt

SHOW_N = 6
img_dir = os.path.join(DATASET_DIR, 'train', 'images')
lbl_dir = os.path.join(DATASET_DIR, 'train', 'labels')

imgs = [f for f in os.listdir(img_dir)
        if os.path.splitext(f)[1].lower() in {'.jpg','.jpeg','.png'}]
random.shuffle(imgs)
imgs = imgs[:SHOW_N]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.patch.set_facecolor('#0d1117')
COLOURS = [(0,220,255),(255,120,0),(0,255,120),(220,0,255),(255,220,0),(0,120,255)]

for ax, fname in zip(axes.flat, imgs):
    img = cv2.imread(os.path.join(img_dir, fname))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    lbl_path = os.path.join(lbl_dir, os.path.splitext(fname)[0] + '.txt')
    if os.path.exists(lbl_path):
        with open(lbl_path) as lf:
            for li, line in enumerate(lf):
                parts = list(map(float, line.strip().split()))
                colour = COLOURS[li % len(COLOURS)]
                if len(parts) == 5:
                    # Bounding box: cls cx cy bw bh (normalised)
                    _, cx, cy, bw, bh = parts
                    x1 = int((cx - bw/2) * w);  y1 = int((cy - bh/2) * h)
                    x2 = int((cx + bw/2) * w);  y2 = int((cy + bh/2) * h)
                    # Draw ellipse fitting the bounding box
                    ex, ey = (x1+x2)//2, (y1+y2)//2
                    ea, eb = (x2-x1)//2, (y2-y1)//2
                    overlay = img.copy()
                    cv2.ellipse(overlay, (ex,ey), (ea,eb), 0, 0, 360, colour, -1)
                    img = cv2.addWeighted(overlay, 0.25, img, 0.75, 0)
                    cv2.ellipse(img, (ex,ey), (ea,eb), 0, 0, 360, colour, 2, cv2.LINE_AA)
                elif len(parts) > 5:
                    # Polygon mask (if available)
                    coords = np.array(parts[1:]).reshape(-1,2)
                    coords[:,0] *= w; coords[:,1] *= h
                    pts = coords.astype(np.int32)
                    overlay = img.copy()
                    cv2.fillPoly(overlay, [pts], colour)
                    img = cv2.addWeighted(overlay, 0.30, img, 0.70, 0)
                    cv2.polylines(img, [pts], True, colour, 2, cv2.LINE_AA)

    ax.imshow(img)
    ax.set_title(fname[:28], color='white', fontsize=8)
    ax.axis('off')

plt.suptitle('Dataset Samples — Cylinder Detection', color='white',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 🚀 Cell 5 — Training YOLOv8n

In [ ]:
from ultralytics import YOLO
import torch

# ── Config ────────────────────────────────────────────────────────────────────
EPOCHS      = 100
IMG_SIZE    = 640
BATCH_SIZE  = 16        # Giảm xuống 8 nếu OOM
DEVICE      = 0 if torch.cuda.is_available() else 'cpu'
PROJECT_DIR = DRIVE_OUTPUT_DIR
RUN_NAME    = 'cylinder_det'

print(f'🔧 Device : {DEVICE}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'   GPU    : {props.name}')
    print(f'   VRAM   : {props.total_memory/1e9:.1f} GB')

print(f'\n📋 Config:')
print(f'   Model   : {BASE_MODEL}')
print(f'   Epochs  : {EPOCHS}  (early stop patience=30)')
print(f'   ImgSize : {IMG_SIZE}')
print(f'   Batch   : {BATCH_SIZE}')
print(f'   Output  : {PROJECT_DIR}/{RUN_NAME}\n')

# ── Load & train ──────────────────────────────────────────────────────────────
model = YOLO(BASE_MODEL)

results = model.train(
    data        = DATA_YAML,
    epochs      = EPOCHS,
    imgsz       = IMG_SIZE,
    batch       = BATCH_SIZE,
    device      = DEVICE,
    project     = PROJECT_DIR,
    name        = RUN_NAME,
    # ── Augmentation ──────────────────────────────────────────────────────
    hsv_h       = 0.015,
    hsv_s       = 0.7,
    hsv_v       = 0.4,
    degrees     = 15.0,
    translate   = 0.1,
    scale       = 0.5,
    shear       = 2.0,
    flipud      = 0.1,
    fliplr      = 0.5,
    mosaic      = 1.0,
    mixup       = 0.15,
    copy_paste  = 0.15,
    # ── Training ──────────────────────────────────────────────────────────
    patience    = 30,
    save        = True,
    save_period = 10,
    plots       = True,
    verbose     = True,
)

BEST_WEIGHTS = f'{PROJECT_DIR}/{RUN_NAME}/weights/best.pt'
print(f'\n✅ Training complete!')
print(f'   Best weights → {BEST_WEIGHTS}')

## 📊 Cell 6 — Validation & biểu đồ

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from ultralytics import YOLO

best_model = YOLO(BEST_WEIGHTS)
print('🔍 Validating ...')
val_res = best_model.val(data=DATA_YAML, imgsz=IMG_SIZE, device=DEVICE)

print('\n📈 Validation metrics:')
try:
    print(f'   mAP50    : {val_res.box.map50:.4f}')
    print(f'   mAP50-95 : {val_res.box.map:.4f}')
    print(f'   Precision: {val_res.box.mp:.4f}')
    print(f'   Recall   : {val_res.box.mr:.4f}')
except Exception:
    print(val_res)

# Hiển thị plots
run_dir = f'{PROJECT_DIR}/{RUN_NAME}'
plot_files = ['results.png','confusion_matrix.png','PR_curve.png','F1_curve.png']
available = [p for p in plot_files if os.path.exists(os.path.join(run_dir, p))]

if available:
    fig, axes = plt.subplots(1, len(available), figsize=(6*len(available), 5))
    if len(available) == 1: axes = [axes]
    fig.patch.set_facecolor('#0d1117')
    for ax, pf in zip(axes, available):
        ax.imshow(mpimg.imread(os.path.join(run_dir, pf)))
        ax.set_title(pf.replace('.png',''), color='white', fontsize=10)
        ax.axis('off')
    plt.tight_layout(); plt.show()

## 🔍 Cell 7 — Inference helpers: Ellipse border + GrabCut contour

In [ ]:
"""
Inference helpers
=================
Vì dataset không có polygon mask, ta có 2 chế độ vẽ border:

  BORDER_MODE = 'ellipse'   → Fit ellipse vào bounding box (nhanh, đẹp với hình trụ)
  BORDER_MODE = 'grabcut'   → Dùng GrabCut segmentation trong box để lấy contour thực
  BORDER_MODE = 'seg'       → Dùng mask polygon (nếu model có seg head)
"""

import cv2, numpy as np

_RNG     = np.random.default_rng(42)
_PALETTE = [tuple(int(c) for c in _RNG.integers(80, 230, 3)) for _ in range(100)]

BORDER_MODE = 'ellipse'   # ← 'ellipse' | 'grabcut'


def cls_colour(cls_id: int) -> tuple:
    return _PALETTE[int(cls_id) % len(_PALETTE)]


# ── Ellipse border ────────────────────────────────────────────────────────────
def draw_ellipse_border(img, x1, y1, x2, y2, colour, fill=True, alpha=0.28):
    """Vẽ ellipse 3 lớp (glow → main → highlight) bao phủ bounding box."""
    cx, cy = (x1+x2)//2, (y1+y2)//2
    rx, ry = max(1,(x2-x1)//2), max(1,(y2-y1)//2)

    if fill:
        overlay = img.copy()
        cv2.ellipse(overlay, (cx,cy), (rx,ry), 0, 0, 360, colour, -1)
        cv2.addWeighted(overlay, alpha, img, 1-alpha, 0, img)

    glow   = tuple(max(0, int(c*0.5))     for c in colour)
    bright = tuple(min(255, int(c*1.8))   for c in colour)

    cv2.ellipse(img, (cx,cy), (rx+4,ry+4), 0, 0, 360, glow,   8, cv2.LINE_AA)  # outer glow
    cv2.ellipse(img, (cx,cy), (rx,ry),     0, 0, 360, colour,  3, cv2.LINE_AA)  # main
    cv2.ellipse(img, (cx,cy), (rx-2,ry-2), 0, 0, 360, bright,  1, cv2.LINE_AA)  # inner highlight

    return cx, cy


# ── GrabCut contour border ────────────────────────────────────────────────────
def draw_grabcut_border(img, x1, y1, x2, y2, colour, fill=True, alpha=0.28):
    """
    Dùng GrabCut để tách foreground trong bounding box,
    sau đó vẽ contour của foreground mask.
    """
    h_img, w_img = img.shape[:2]
    # Clamp + margin
    margin = 4
    gx1 = max(0, x1 - margin);  gy1 = max(0, y1 - margin)
    gx2 = min(w_img-1, x2 + margin);  gy2 = min(h_img-1, y2 + margin)
    bw, bh = gx2 - gx1, gy2 - gy1

    if bw < 10 or bh < 10:
        return draw_ellipse_border(img, x1, y1, x2, y2, colour, fill, alpha)

    rect  = (gx1, gy1, bw, bh)
    mask  = np.zeros(img.shape[:2], np.uint8)
    bgdModel = np.zeros((1,65), np.float64)
    fgdModel = np.zeros((1,65), np.float64)

    try:
        cv2.grabCut(img, mask, rect, bgdModel, fgdModel, 4, cv2.GC_INIT_WITH_RECT)
    except Exception:
        return draw_ellipse_border(img, x1, y1, x2, y2, colour, fill, alpha)

    fg_mask = np.where((mask == 2) | (mask == 0), 0, 1).astype(np.uint8)
    contours, _ = cv2.findContours(fg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return draw_ellipse_border(img, x1, y1, x2, y2, colour, fill, alpha)

    biggest = max(contours, key=cv2.contourArea)

    if fill:
        overlay = img.copy()
        cv2.fillPoly(overlay, [biggest], colour)
        cv2.addWeighted(overlay, alpha, img, 1-alpha, 0, img)

    glow   = tuple(max(0, int(c*0.5))   for c in colour)
    bright = tuple(min(255,int(c*1.8))  for c in colour)
    cv2.drawContours(img, [biggest], -1, glow,   7, cv2.LINE_AA)
    cv2.drawContours(img, [biggest], -1, colour, 3, cv2.LINE_AA)
    cv2.drawContours(img, [biggest], -1, bright, 1, cv2.LINE_AA)

    M = cv2.moments(biggest)
    if M['m00'] != 0:
        return int(M['m10']/M['m00']), int(M['m01']/M['m00'])
    return (x1+x2)//2, (y1+y2)//2


# ── Label helper ──────────────────────────────────────────────────────────────
def draw_label(img, lines, anchor, colour):
    font, scale, thick, pad, lh = cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1, 5, 20
    tw  = max(cv2.getTextSize(l, font, scale, thick)[0][0] for l in lines)
    bh  = lh*len(lines) + pad*2
    ax, ay = anchor
    H, W   = img.shape[:2]
    ay = max(0, min(ay, H-bh));  ax = max(0, min(ax, W-tw-pad*2))
    bg = tuple(max(0,int(c*0.3)) for c in colour)
    cv2.rectangle(img,(ax,ay),(ax+tw+pad*2,ay+bh),bg,-1)
    cv2.rectangle(img,(ax,ay),(ax+tw+pad*2,ay+bh),colour,1)
    for i, ln in enumerate(lines):
        ty = ay + pad + lh*(i+1) - 2
        cv2.putText(img, ln, (ax+pad,ty), font, scale, (255,255,255), thick, cv2.LINE_AA)


# ── Main inference function ───────────────────────────────────────────────────
def infer_frame(model, frame, conf=0.35, fill=True, alpha=0.28, mode=None):
    """
    Chạy YOLO inference và vẽ border shape.
    mode: 'ellipse' | 'grabcut' | 'seg' (auto-detect từ model nếu None)
    """
    if mode is None:
        mode = BORDER_MODE

    results = model(frame, conf=conf, verbose=False)
    out     = frame.copy()
    boxes   = results[0].boxes
    masks   = results[0].masks   # None nếu là detect model
    n_det   = len(boxes) if boxes is not None else 0

    if boxes is not None and n_det > 0:
        for i, box in enumerate(boxes):
            x1,y1,x2,y2 = (int(v) for v in box.xyxy[0].tolist())
            cls_id = int(box.cls[0])
            conf_v = float(box.conf[0])
            name   = model.names[cls_id]
            colour = cls_colour(cls_id)

            # ── Chọn phương pháp vẽ border ──────────────────────────────────
            if mode == 'seg' and masks is not None and i < len(masks):
                # True segmentation mask (nếu có)
                xy = masks[i].xy
                if xy is not None and len(xy)>0 and len(xy[0])>=3:
                    poly   = xy[0].astype(np.int32)
                    pts    = poly.reshape((-1,1,2))
                    if fill:
                        ov = out.copy()
                        cv2.fillPoly(ov,[pts],colour)
                        cv2.addWeighted(ov,alpha,out,1-alpha,0,out)
                    glow   = tuple(max(0,int(c*0.5))  for c in colour)
                    bright = tuple(min(255,int(c*1.8)) for c in colour)
                    cv2.polylines(out,[pts],True,glow,  7,cv2.LINE_AA)
                    cv2.polylines(out,[pts],True,colour,3,cv2.LINE_AA)
                    cv2.polylines(out,[pts],True,bright,1,cv2.LINE_AA)
                    M  = cv2.moments(pts)
                    cx = int(M['m10']/M['m00']) if M['m00'] else (x1+x2)//2
                    cy = int(M['m01']/M['m00']) if M['m00'] else (y1+y2)//2
                else:
                    cx, cy = draw_ellipse_border(out,x1,y1,x2,y2,colour,fill,alpha)

            elif mode == 'grabcut':
                cx, cy = draw_grabcut_border(out,x1,y1,x2,y2,colour,fill,alpha)

            else:   # 'ellipse' (default)
                cx, cy = draw_ellipse_border(out,x1,y1,x2,y2,colour,fill,alpha)

            # ── Crosshair ────────────────────────────────────────────────────
            arm = 12
            cv2.line(out,(cx-arm,cy),(cx+arm,cy),(0,255,80),2,cv2.LINE_AA)
            cv2.line(out,(cx,cy-arm),(cx,cy+arm),(0,255,80),2,cv2.LINE_AA)
            cv2.circle(out,(cx,cy),4,(0,255,80),-1)

            # ── Label ────────────────────────────────────────────────────────
            draw_label(out,
                       [f'{name}  {conf_v:.0%}', f'ctr:({cx},{cy}) px'],
                       (x1, max(0,y1-46)),
                       colour)

    return out, n_det


print('✅ Inference helpers loaded.')
print(f'   Border mode : {BORDER_MODE}')
print('   Thay đổi BORDER_MODE = "grabcut" để dùng GrabCut contour.')

## 🖼️ Cell 8 — Inference trên ảnh test set

In [ ]:
import os, random
import cv2, numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

CONF_THRESH  = 0.35
MASK_FILL    = True
MASK_ALPHA   = 0.28
SHOW_N       = 6
# BORDER_MODE đã set ở Cell 7 — override nếu muốn:
# BORDER_MODE = 'grabcut'

best_model = YOLO(BEST_WEIGHTS)

test_dir = os.path.join(DATASET_DIR, 'test', 'images')
if not os.path.exists(test_dir):
    test_dir = os.path.join(DATASET_DIR, 'valid', 'images')

test_imgs = [os.path.join(test_dir, f)
             for f in os.listdir(test_dir)
             if os.path.splitext(f)[1].lower() in {'.jpg','.jpeg','.png'}]
random.shuffle(test_imgs)
test_imgs = test_imgs[:SHOW_N]

cols = 3
rows = (len(test_imgs)+cols-1)//cols
fig, axes = plt.subplots(rows, cols, figsize=(7*cols, 5*rows))
fig.patch.set_facecolor('#0d1117')
axes_flat = np.array(axes).flatten()

for ax, ip in zip(axes_flat, test_imgs):
    bgr  = cv2.imread(ip)
    ann, nd = infer_frame(best_model, bgr, CONF_THRESH, MASK_FILL, MASK_ALPHA)
    ax.imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{os.path.basename(ip)}  [{nd} det]', color='white', fontsize=8)
    ax.axis('off')

for ax in axes_flat[len(test_imgs):]:
    ax.axis('off')

mode_label = BORDER_MODE if 'BORDER_MODE' in dir() else 'ellipse'
plt.suptitle(f'Cylinder Detection — {mode_label} border  (conf≥{CONF_THRESH})',
             color='white', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
print(f'✅ Done — {len(test_imgs)} images.')

## 📤 Cell 9 — Test ảnh upload tuỳ chọn

In [ ]:
from google.colab import files
import cv2, numpy as np
import matplotlib.pyplot as plt

CONF_THRESH = 0.35   # ← Chỉnh tự do
MASK_FILL   = True
MASK_ALPHA  = 0.28

print('📁 Chọn ảnh JPG/PNG để test ...')
uploaded = files.upload()

for filename, data in uploaded.items():
    img_np = np.frombuffer(data, np.uint8)
    bgr    = cv2.imdecode(img_np, cv2.IMREAD_COLOR)
    print(f'\n🔍 {filename}  ({bgr.shape[1]}×{bgr.shape[0]})')

    ann, nd = infer_frame(best_model, bgr, CONF_THRESH, MASK_FILL, MASK_ALPHA)
    rgb = cv2.cvtColor(ann, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 7))
    plt.imshow(rgb)
    plt.title(f'{filename}  —  {nd} cylinder(s) detected', fontsize=12)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    out_path = f'/content/result_{filename}'
    cv2.imwrite(out_path, ann)
    files.download(out_path)
    print(f'   ✅ Saved & downloaded → {out_path}')

## 🎬 Cell 10 — Test video upload

In [ ]:
from google.colab import files
from IPython.display import HTML, display as ipy_display
import base64, cv2, os, time
import numpy as np

CONF_THRESH = 0.35
MASK_FILL   = True
MASK_ALPHA  = 0.28
FRAME_SKIP  = 0      # 0 = xử lý tất cả frame

print('📁 Chọn video (MP4/AVI) ...')
uploaded_vid = files.upload()

for filename, data in uploaded_vid.items():
    vid_path = f'/content/{filename}'
    with open(vid_path, 'wb') as vf:
        vf.write(data)

    cap   = cv2.VideoCapture(vid_path)
    fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
    W     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out_path = f'/content/result_{filename}'
    writer   = cv2.VideoWriter(out_path,
                               cv2.VideoWriter_fourcc(*'mp4v'),
                               fps, (W, H))

    print(f'\n🎬 {filename} — {W}×{H} @ {fps:.0f}fps  ({total} frames) ...')
    t0, fi, skip = time.time(), 0, 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        fi += 1
        if FRAME_SKIP > 0:
            skip += 1
            if skip % (FRAME_SKIP+1) != 0:
                writer.write(frame)
                continue

        ann, nd = infer_frame(best_model, frame, CONF_THRESH, MASK_FILL, MASK_ALPHA)

        # HUD
        info = f'Frame {fi}/{total}  Det:{nd}'
        cv2.putText(ann, info, (10,28), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0,0,0),   3, cv2.LINE_AA)
        cv2.putText(ann, info, (10,28), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (180,255,180), 1, cv2.LINE_AA)
        writer.write(ann)

        if fi % 50 == 0:
            pct = fi/total*100
            print(f'   {fi}/{total} ({pct:.0f}%)  elapsed:{time.time()-t0:.0f}s  det:{nd}')

    cap.release(); writer.release()
    print(f'\n✅ Output → {out_path}')

    # Re-encode H264 cho browser
    h264 = out_path.replace('.mp4','_h264.mp4')
    os.system(f'ffmpeg -y -i {out_path} -vcodec libx264 {h264} -loglevel quiet')
    display_path = h264 if os.path.exists(h264) else out_path

    with open(display_path,'rb') as vf:
        b64 = base64.b64encode(vf.read()).decode()

    ipy_display(HTML(f'''
    <video width="800" controls>
      <source src="data:video/mp4;base64,{b64}" type="video/mp4">
    </video>
    '''))
    files.download(display_path)

## 📦 Cell 11 — Export & Download model

In [ ]:
from ultralytics import YOLO
from google.colab import files
import os

EXPORT_FORMAT = 'onnx'   # 'onnx' | 'torchscript' | 'tflite' | 'engine'

best_model = YOLO(BEST_WEIGHTS)
print(f'📤 Exporting to {EXPORT_FORMAT.upper()} ...')
exported = best_model.export(format=EXPORT_FORMAT, imgsz=IMG_SIZE)
print(f'✅ Exported → {exported}')

print('\n⬇️  Downloading best.pt ...')
files.download(BEST_WEIGHTS)

print(f'⬇️  Downloading {os.path.basename(str(exported))} ...')
files.download(str(exported))

---
## 📝 Tóm tắt

### Vì sao dùng Ellipse thay vì Bounding Box?

| Phương pháp | Mô tả | Ưu điểm |
|---|---|---|
| **Ellipse** *(default)* | Fit ellipse vào bounding box | Nhanh, phù hợp hoàn toàn với hình trụ |
| **GrabCut** | Segment foreground trong box → lấy contour | Chính xác hơn, chậm hơn |
| **Seg mask** | Dùng khi có polygon annotations | Tốt nhất, cần dataset có mask |

### Thay đổi chế độ border
```python
BORDER_MODE = 'ellipse'   # nhanh, đẹp với hình trụ
BORDER_MODE = 'grabcut'   # contour thực, chậm hơn ~5×
```

### Thứ tự chạy
Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8/9/10/11